# ============================================================
# УЛУЧШЕННЫЙ CLTV NOTEBOOK - ДВА СЕГМЕНТА
# Часть 1: Импорты, конфигурация, загрузка данных из Parquet
# ============================================================

In [ ]:
# %% ИМПОРТЫ И НАСТРОЙКИ
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from dateutil.relativedelta import relativedelta
import os
import warnings
from collections import defaultdict, deque
from scipy import stats
import logging
import pickle
import json
from pathlib import Path

from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
import optuna
from optuna.samplers import TPESampler

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')
plt.style.use('default')
sns.set_palette("husl")

print("Импорты загружены")

In [ ]:
# %% КОНФИГУРАЦИЯ
class Config:
    # Пути к Parquet файлам
    DATA_DIR = Path("data")
    TRAIN_PATH = DATA_DIR / "train_data.parquet"
    PROD_PATH = DATA_DIR / "prod_data.parquet"
    CHURN_PATH = DATA_DIR / "churn_data.parquet"
    
    MODEL_DIR = Path("models")
    MODEL_VERSION = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    FORECAST_START = "2025-10-31"
    HORIZON_MONTHS = 6
    DISCOUNT_RATE_ANNUAL = 0.12
    VALIDATION_CUTOFF = "2025-03-31"
    MIN_SAMPLES_PER_SEGMENT = 1000
    
    CATEGORICAL_FEATURES = ['QUALITY_CODE', 'SUBJECT_KIND_ID', 'EC_SECTOR_ID']
    BASE_FEATURES = [
        'MARGIN', 'MARGIN_LAG1', 'MARGIN_LAG2', 'MARGIN_LAG3',
        'MARGIN_AVG_1M_LAG', 'MARGIN_AVG_2M_LAG', 'MARGIN_AVG_3M_LAG',
        'MARGIN_AVG_6M_LAG', 'MARGIN_AVG_12M_LAG', 'MARGIN_STDDEV_12M_LAG',
        'MARGIN_GROWTH_RATE_3M', 'MONTH_OF_YEAR', 'QUARTER_OF_YEAR', 'TENURE_MONTHS'
    ]
    
    OPTUNA_TRIALS = 10
    OPTUNA_TIMEOUT = 1800
    CV_SPLITS = 3
    TRAIN_QUANTILES = False 
    
    # Маппинг сегментов
    SEGMENT_MAPPING = {
        '1026': 'small',      # MICRO
        '1027': 'small',      # SMALL
        '1040': 'small',      # Дополнительный малый сегмент
        '1022': 'large_and_middle',  # MIDDLE
        '1023': 'large_and_middle',  # LARGE
    }
    
    @classmethod
    def ensure_directories(cls):
        cls.MODEL_DIR.mkdir(parents=True, exist_ok=True)
        (cls.MODEL_DIR / cls.MODEL_VERSION).mkdir(parents=True, exist_ok=True)

Config.ensure_directories()
print(f"Версия: {Config.MODEL_VERSION}")
print(f"\nМаппинг сегментов:")
for old_seg, new_seg in Config.SEGMENT_MAPPING.items():
    print(f"  {old_seg} → {new_seg}")

In [ ]:
# %% ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
def add_months(dt_str, k):
    d = datetime.strptime(dt_str, "%Y-%m-%d")
    return (d + relativedelta(months=+k)).strftime("%Y-%m-%d")

def discounted(value, months, annual_rate):
    if annual_rate <= 0:
        return value
    monthly_rate = (1 + annual_rate) ** (1/12) - 1
    return value / ((1 + monthly_rate) ** months)

def read_table(path):
    """Чтение Parquet файлов"""
    if not os.path.exists(path):
        logger.warning(f"Файл не найден: {path}")
        return pd.DataFrame()
    
    ext = os.path.splitext(path)[1].lower()
    if ext in [".parquet", ".pq", ".parq"]:
        return pd.read_parquet(path)
    else:
        raise ValueError(f"Ожидается Parquet файл, получен: {ext}")

def fix_categorical_features(df, cat_features):
    df_fixed = df.copy()
    for col in cat_features:
        if col in df_fixed.columns:
            df_fixed[col] = df_fixed[col].fillna('UNKNOWN').astype(str)
            df_fixed[col] = df_fixed[col].str.replace('.0', '', regex=False)
    return df_fixed

def stabilize_target(y):
    return np.sign(y) * np.log1p(np.abs(y))

def inverse_stabilize_target(y_stable):
    return np.sign(y_stable) * (np.exp(np.abs(y_stable)) - 1)

def calculate_business_metrics(y_true, y_pred, percentiles=[0.5, 0.75, 0.9, 0.95]):
    metrics = {}
    for p in percentiles:
        threshold = np.quantile(y_true, p)
        mask = y_true >= threshold
        if mask.sum() > 0:
            mae = mean_absolute_error(y_true[mask], y_pred[mask])
            r2 = r2_score(y_true[mask], y_pred[mask])
            metrics[f'mae_top_{int((1-p)*100)}pct'] = mae
            metrics[f'r2_top_{int((1-p)*100)}pct'] = r2
    return metrics

print("Функции загружены")

In [ ]:
# %% ЗАГРУЗКА ДАННЫХ ИЗ PARQUET
logger.info("Загрузка данных из Parquet...")

train = read_table(Config.TRAIN_PATH)
prod = read_table(Config.PROD_PATH)
churn_raw = read_table(Config.CHURN_PATH)

print(f"Обучающая: {len(train):,} записей, {train['CLIENT_ID'].nunique():,} клиентов")
print(f"Продакшн: {len(prod):,} записей")
print(f"Churn: {len(churn_raw):,} записей")

if train.empty or prod.empty or churn_raw.empty:
    print("\n⚠️  ВНИМАНИЕ: Некоторые файлы не загружены!")
    print("Пожалуйста, запустите сначала notebook 'data_loader.ipynb'")

In [ ]:
# %% ОБЪЕДИНЕНИЕ СЕГМЕНТОВ
logger.info("Объединение сегментов...")

print("\nОригинальные сегменты:")
print(train['SEGMENT_ID'].value_counts().sort_index())

# Преобразование SEGMENT_ID в строку для маппинга
train['SEGMENT_ID'] = train['SEGMENT_ID'].astype(str)
prod['SEGMENT_ID'] = prod['SEGMENT_ID'].astype(str)

# Применение маппинга
train['SEGMENT_ID'] = train['SEGMENT_ID'].map(Config.SEGMENT_MAPPING)
prod['SEGMENT_ID'] = prod['SEGMENT_ID'].map(Config.SEGMENT_MAPPING)

# Обработка неизвестных сегментов (если есть)
train['SEGMENT_ID'] = train['SEGMENT_ID'].fillna('small')
prod['SEGMENT_ID'] = prod['SEGMENT_ID'].fillna('small')

print("\nНовые объединенные сегменты:")
print(train['SEGMENT_ID'].value_counts().sort_index())

print("\nРаспределение в train:")
for segment in sorted(train['SEGMENT_ID'].unique()):
    count = len(train[train['SEGMENT_ID'] == segment])
    clients = train[train['SEGMENT_ID'] == segment]['CLIENT_ID'].nunique()
    pct = 100 * count / len(train)
    print(f"  {segment}: {count:,} записей ({pct:.1f}%), {clients:,} клиентов")

print("\nРаспределение в prod:")
for segment in sorted(prod['SEGMENT_ID'].unique()):
    count = len(prod[prod['SEGMENT_ID'] == segment])
    pct = 100 * count / len(prod)
    print(f"  {segment}: {count:,} записей ({pct:.1f}%)")

In [ ]:
# %% ОБРАБОТКА CHURN
def calculate_monthly_hazard_rate(churn_3m_prob):
    churn_3m = float(max(0.0, min(0.99999, churn_3m_prob)))
    survival_3m = 1.0 - churn_3m
    if survival_3m <= 0:
        monthly_hazard = 0.5
    else:
        monthly_hazard = -np.log(survival_3m) / 3.0
    return {
        'monthly_hazard': monthly_hazard,
        'monthly_churn_prob': 1.0 - np.exp(-monthly_hazard),
        'implied_3m_survival': np.exp(-3 * monthly_hazard),
        'original_3m_churn': churn_3m
    }

churn = churn_raw.copy()
churn_map = {}
for _, row in churn.iterrows():
    churn_map[row['CLIENT_ID']] = calculate_monthly_hazard_rate(row['CHURN_PROB_3M'])

print(f"Churn map: {len(churn_map):,} клиентов")

def calculate_survival_probability(client_id, churn_map):
    if client_id not in churn_map:
        return 1.0
    return np.exp(-churn_map[client_id]['monthly_hazard'])

In [ ]:
# %% ПРЕДОБРАБОТКА
all_categorical = Config.CATEGORICAL_FEATURES + ['SEGMENT_ID']
train_fixed = fix_categorical_features(train, all_categorical)
prod_fixed = fix_categorical_features(prod, all_categorical)

ALL_FEATURES = ['SEGMENT_ID'] + Config.BASE_FEATURES + Config.CATEGORICAL_FEATURES
available_features = [f for f in ALL_FEATURES if f in train_fixed.columns]

numeric_features = [f for f in available_features if f not in all_categorical]
for col in numeric_features:
    train_fixed[col] = train_fixed[col].fillna(0.0)
    if col in prod_fixed.columns:
        prod_fixed[col] = prod_fixed[col].fillna(0.0)

print(f"Фичи готовы: {len(available_features)}")
print(f"Сегменты: {sorted(train_fixed['SEGMENT_ID'].unique())}")

In [ ]:
# %% КЛАСС УЛУЧШЕННОГО CLTV ДЛЯ ДВУХ СЕГМЕНТОВ
class ImprovedSegmentedCLTV:
    """Улучшенная версия с Optuna, квантилями и сохранением для двух сегментов"""
    
    def __init__(self, min_samples_per_segment=1000, use_optuna=True, 
                 n_trials=30, cv_splits=3):
        self.models = {}
        self.quantile_models = {}
        self.segment_stats = {}
        self.min_samples_per_segment = min_samples_per_segment
        self.fallback_segments = {}
        self.feature_importance = {}
        self.use_optuna = use_optuna
        self.n_trials = n_trials
        self.cv_splits = cv_splits
        self.best_params = {}
        self.metadata = {}
        
    def analyze_segment_distribution(self, df, segment_col='SEGMENT_ID'):
        segment_stats = df.groupby(segment_col).agg({
            'CLIENT_ID': 'nunique',
            'TARGET_NEXT_MARGIN': ['count', 'mean', 'std', 'min', 'max']
        }).round(2)
        
        segment_stats.columns = ['unique_clients', 'total_records', 'avg_margin', 
                               'std_margin', 'min_margin', 'max_margin']
        segment_stats['records_per_client'] = (
            segment_stats['total_records'] / segment_stats['unique_clients']
        ).round(1)
        
        small = segment_stats[segment_stats['total_records'] < self.min_samples_per_segment].index.tolist()
        large = segment_stats[segment_stats['total_records'] >= self.min_samples_per_segment].index.tolist()
        
        print("Анализ сегментов:")
        print(segment_stats)
        print(f"\nБольшие сегменты (>={self.min_samples_per_segment}): {large}")
        print(f"Малые сегменты (<{self.min_samples_per_segment}): {small}")
        
        return segment_stats, large, small
    
    def prepare_segment_data(self, df, segment_id, features, target_col, validation_cutoff):
        segment_data = df[df['SEGMENT_ID'] == segment_id].copy()
        segment_data['target_stable'] = stabilize_target(segment_data[target_col])
        
        train_mask = pd.to_datetime(segment_data['MONTH_END']) <= pd.to_datetime(validation_cutoff)
        val_mask = ~train_mask
        
        features_for_segment = [f for f in features if f != 'SEGMENT_ID']
        
        X_train = segment_data[train_mask][features_for_segment]
        y_train = segment_data[train_mask]['target_stable']
        X_val = segment_data[val_mask][features_for_segment] 
        y_val = segment_data[val_mask]['target_stable']
        y_val_original = segment_data[val_mask][target_col]
        
        return X_train, y_train, X_val, y_val, y_val_original, segment_data
    
    def optimize_hyperparameters_optuna(self, segment_id, X_train, y_train, X_val, y_val, cat_features):
        cat_indices = []
        features_list = X_train.columns.tolist()
        for cat_feat in cat_features:
            if cat_feat in features_list:
                cat_indices.append(features_list.index(cat_feat))
        
        def objective(trial):
            params = {
                'iterations': trial.suggest_int('iterations', 500, 3000),
                'depth': trial.suggest_int('depth', 4, 8),
                'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.1, log=True),
                'l2_leaf_reg': trial.suggest_int('l2_leaf_reg', 1, 100),
                'random_seed': 42,
                'loss_function': 'MAE',
                'verbose': False,
                'early_stopping_rounds': 50
            }
            
            train_pool = Pool(X_train, y_train, cat_features=cat_indices)
            val_pool = Pool(X_val, y_val, cat_features=cat_indices)
            
            model = CatBoostRegressor(**params)
            model.fit(train_pool, eval_set=val_pool, use_best_model=True)
            
            return mean_absolute_error(y_val, model.predict(X_val))
        
        study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=42))
        study.optimize(objective, n_trials=self.n_trials, timeout=Config.OPTUNA_TIMEOUT, 
                      show_progress_bar=True)
        
        print(f"  Лучший MAE: {study.best_value:.3f}")
        return study.best_params
    
    def get_default_params(self, segment_id, data_size):
        base = {"random_seed": 42, "loss_function": "MAE", "verbose": False, 
                "early_stopping_rounds": 50}
        if data_size > 100000:
            base.update({"iterations": 2000, "depth": 6, "learning_rate": 0.01, "l2_leaf_reg": 20})
        elif data_size > 10000:
            base.update({"iterations": 1500, "depth": 5, "learning_rate": 0.015, "l2_leaf_reg": 30})
        else:
            base.update({"iterations": 1000, "depth": 4, "learning_rate": 0.02, "l2_leaf_reg": 50})
        return base
    
    def train_segment_model(self, segment_id, X_train, y_train, X_val, y_val, categorical_features=None):
        cat_indices = []
        if categorical_features:
            features_list = X_train.columns.tolist()
            for cat_feat in categorical_features:
                if cat_feat in features_list:
                    cat_indices.append(features_list.index(cat_feat))
        
        if self.use_optuna:
            print(f"  Optuna ({self.n_trials} trials)...")
            best_params = self.optimize_hyperparameters_optuna(
                segment_id, X_train, y_train, X_val, y_val, categorical_features
            )
            best_params.update({'random_seed': 42, 'loss_function': 'MAE', 'verbose': False})
        else:
            best_params = self.get_default_params(segment_id, len(X_train))
        
        self.best_params[segment_id] = best_params
        
        train_pool = Pool(X_train, y_train, cat_features=cat_indices)
        val_pool = Pool(X_val, y_val, cat_features=cat_indices)
        
        model = CatBoostRegressor(**best_params)
        model.fit(train_pool, eval_set=val_pool, use_best_model=True)
        
        # Квантильные модели
        quantile_models = {}
        if Config.TRAIN_QUANTILES:
            print(f"  Обучение квантилей (0.1, 0.5, 0.9)...")
            for quantile in [0.1, 0.5, 0.9]:
                q_params = best_params.copy()
                q_params['loss_function'] = f'Quantile:alpha={quantile}'
                q_model = CatBoostRegressor(**q_params)
                q_model.fit(train_pool, eval_set=val_pool, use_best_model=True)
                quantile_models[quantile] = q_model
        else:
            print(f"  Квантили отключены")
        
        train_pred = model.predict(X_train)
        val_pred = model.predict(X_val)
        
        metrics = {
            'train_r2': r2_score(y_train, train_pred),
            'val_r2': r2_score(y_val, val_pred),
            'val_mae': mean_absolute_error(y_val, val_pred),
            'train_samples': len(X_train),
            'val_samples': len(X_val),
            'feature_importance': pd.DataFrame({
                'feature': X_train.columns,
                'importance': model.feature_importances_
            }).sort_values('importance', ascending=False),
            'business_metrics': calculate_business_metrics(y_val, val_pred),
            'best_params': best_params
        }
        
        return model, quantile_models, metrics
    
    def predict_segment(self, segment_id, X, return_stable=False, return_uncertainty=False):
        if segment_id in self.models:
            model = self.models[segment_id]
            X_pred = X.drop('SEGMENT_ID', axis=1) if 'SEGMENT_ID' in X.columns and segment_id != 'FALLBACK' else X
        elif segment_id in self.fallback_segments:
            model = self.models[self.fallback_segments[segment_id]]
            X_pred = X
        else:
            raise ValueError(f"Нет модели для {segment_id}")
        
        pred_stable = model.predict(X_pred)
        result = {'prediction': pred_stable}
        
        if return_uncertainty:
            if segment_id in self.quantile_models and len(self.quantile_models[segment_id]) > 0:
                q_models = self.quantile_models[segment_id]
                result['lower_bound'] = q_models[0.1].predict(X_pred)
                result['median'] = q_models[0.5].predict(X_pred)
                result['upper_bound'] = q_models[0.9].predict(X_pred)
            else:
                result['lower_bound'] = pred_stable * 0.7
                result['median'] = pred_stable
                result['upper_bound'] = pred_stable * 1.3
        
        if not return_stable:
            result['prediction'] = inverse_stabilize_target(result['prediction'])
            if 'lower_bound' in result:
                result['lower_bound'] = inverse_stabilize_target(result['lower_bound'])
                result['median'] = inverse_stabilize_target(result['median'])
                result['upper_bound'] = inverse_stabilize_target(result['upper_bound'])
        
        return result['prediction'] if not return_uncertainty else result
    
    def save_models(self, path, version, features, churn_map):
        save_path = Path(path) / version
        save_path.mkdir(parents=True, exist_ok=True)
        
        metadata = {
            'version': version,
            'train_date': datetime.now().isoformat(),
            'segments': list(self.models.keys()),
            'features': features,
            'fallback_segments': self.fallback_segments,
            'best_params': self.best_params
        }
        
        metrics_data = {}
        for seg_id, stats in self.segment_stats.items():
            metrics_data[seg_id] = {
                'train_r2': float(stats['train_r2']),
                'val_r2': float(stats['val_r2']),
                'val_mae': float(stats['val_mae']),
                'train_samples': int(stats['train_samples']),
                'val_samples': int(stats['val_samples']),
                'business_metrics': stats['business_metrics']
            }
        metadata['segment_metrics'] = metrics_data
        
        for seg_id, model in self.models.items():
            model.save_model(str(save_path / f"model_seg_{seg_id}.cbm"))
        
        for seg_id, q_models in self.quantile_models.items():
            for q, q_model in q_models.items():
                q_model.save_model(str(save_path / f"model_seg_{seg_id}_q{q}.cbm"))
        
        with open(save_path / "metadata.json", 'w') as f:
            json.dump(metadata, f, indent=2)
        
        for seg_id, importance_df in self.feature_importance.items():
            importance_df.to_csv(save_path / f"feature_importance_{seg_id}.csv", index=False)
        
        with open(save_path / "churn_map.pkl", 'wb') as f:
            pickle.dump(churn_map, f)
        
        with open(save_path / "model_object.pkl", 'wb') as f:
            pickle.dump(self, f)
        
        logger.info(f"Модели сохранены: {save_path}")
        return save_path
    
    @classmethod
    def load_models(cls, path, version):
        load_path = Path(path) / version
        if not load_path.exists():
            raise ValueError(f"Не найдено: {load_path}")
        
        with open(load_path / "model_object.pkl", 'rb') as f:
            model_obj = pickle.load(f)
        
        for seg_id in model_obj.models.keys():
            model_path = load_path / f"model_seg_{seg_id}.cbm"
            if model_path.exists():
                model_obj.models[seg_id] = CatBoostRegressor()
                model_obj.models[seg_id].load_model(str(model_path))
        
        logger.info(f"Модели загружены: {load_path}")
        return model_obj

print("Класс ImprovedSegmentedCLTV создан")

# ============================================================
# ЧАСТЬ 2: КЛАСС УЛУЧШЕННОГО СЕГМЕНТИРОВАННОГО CLTV
# ============================================================